# setup

In [1]:
import os
import sys

os.environ["JAVA_HOME"] = "C:\\Program Files\\Eclipse Adoptium\\jdk-17.0.18.8-hotspot"
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark import SparkContext, SparkConf

conf = SparkConf().setMaster("local[*]").set("spark.driver.memory", "30g")
sc = SparkContext(conf=conf)
print(sc.version)

4.1.1


In [2]:
import multiprocessing
print(multiprocessing.cpu_count())

8


# 1. read the 2 CSVs using spark

In [3]:
%%time
import csv
import io

def parse_csv_line(line):
    """Parse a single CSV line, handling quoted fields correctly."""
    return next(csv.reader(io.StringIO(line)))

# Read matches CSV
matches_raw = sc.textFile("battlesStaging_12072020_to_12262020_WL_tagged.csv")

# Extract header and broadcast it
matches_header = matches_raw.first()
matches_header_bc = sc.broadcast(matches_header)

# Parse rows into dicts, skipping the header
matches_rdd = (
    matches_raw
    .filter(lambda line: line != matches_header_bc.value)
    .map(parse_csv_line)
    .map(lambda fields: dict(zip(matches_header_bc.value.split(","), fields)))
)

CPU times: total: 0 ns
Wall time: 3.35 s


In [4]:
%%time
# Read cards CSV
cards_raw = sc.textFile("CardMasterListSeason18_12082020.csv")

cards_header = cards_raw.first()
cards_header_bc = sc.broadcast(cards_header)

# Build a broadcast dict: card_id -> card_name
# The cards CSV uses columns: team.card1.id and team.card1.name
cards_dict = (
    cards_raw
    .filter(lambda line: line != cards_header_bc.value)
    .map(parse_csv_line)
    .map(lambda fields: dict(zip(cards_header_bc.value.split(","), fields)))
    .map(lambda row: (row["team.card1.id"], row["team.card1.name"]))
    .collectAsMap()
)
cards_bc = sc.broadcast(cards_dict)

CPU times: total: 0 ns
Wall time: 4.88 s


# 2. replace card IDs with card names using Spark Core RDD (distributed computing)

In [5]:
%%time
def add_card_names(row):
    """Look up and attach card names for all 16 card ID columns."""
    lookup = cards_bc.value
    for side in ("winner", "loser"):
        for i in range(1, 9):
            card_id = row.get(f"{side}.card{i}.id", "")
            row[f"{side}.card{i}.name"] = lookup.get(card_id, None)
    return row

result_rdd = matches_rdd.map(add_card_names)
result_rdd.cache()

count = result_rdd.count()
print("Done!", count, "rows")

# Show first row (mirroring df_result.show(1))
first = result_rdd.first()
for k, v in first.items():
    print(f"  {k}: {v}")

Done! 16795959 rows
  : 0
  battleTime: 2020-12-07 07:00:00+00:00
  arena.id: 54000049.0
  gameMode.id: 72000201.0
  average.startingTrophies: 6590.0
  winner.tag: #28RR8PJP0
  winner.startingTrophies: 6581.0
  winner.trophyChange: 31.0
  winner.crowns: 2.0
  winner.kingTowerHitPoints: 4768.0
  winner.princessTowersHitPoints: [1218]
  winner.clan.tag: #YYL2GJUC
  winner.clan.badgeId: 16000111.0
  loser.tag: #9UU8P008P
  loser.startingTrophies: 6599.0
  loser.trophyChange: -31.0
  loser.crowns: 1.0
  loser.kingTowerHitPoints: 147.0
  loser.clan.tag: #P2PUU9JV
  loser.clan.badgeId: 16000119.0
  loser.princessTowersHitPoints: 
  tournamentTag: 
  winner.card1.id: 26000036
  winner.card1.level: 13
  winner.card2.id: 28000015
  winner.card2.level: 13
  winner.card3.id: 26000050
  winner.card3.level: 13
  winner.card4.id: 26000044
  winner.card4.level: 13
  winner.card5.id: 26000054
  winner.card5.level: 13
  winner.card6.id: 28000016
  winner.card6.level: 13
  winner.card7.id: 26000043
  wi

# 3. count wins and losses for all cards

In [6]:
%%time
# Emit (card_name, (wins, losses)) for every card slot in every match.
# Winner slots contribute (1, 0); loser slots contribute (0, 1).
def emit_card_pairs(row):
    pairs = []
    for i in range(1, 9):
        winner_card = row.get(f"winner.card{i}.name")
        loser_card  = row.get(f"loser.card{i}.name")
        if winner_card:
            pairs.append((winner_card, (1, 0)))  # 1 win, 0 losses
        if loser_card:
            pairs.append((loser_card,  (0, 1)))  # 0 wins, 1 loss
    return pairs

# flatMap -> reduceByKey to sum (wins, losses) per card
card_stats_rdd = (
    result_rdd
    .flatMap(emit_card_pairs)
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
    # Restructure to (card_name, win_count, loss_count)
    .map(lambda kv: (kv[0], kv[1][0], kv[1][1]))
    # Sort by win_count descending
    .sortBy(lambda x: -x[1])
)
card_stats_rdd.cache()

# Show top 10 (mirroring df_card_stats.show(10))
print(f"{'card_name':<25} {'win_count':>10} {'loss_count':>12}")
print("-" * 50)
for card_name, win_count, loss_count in card_stats_rdd.take(10):
    print(f"{card_name:<25} {win_count:>10} {loss_count:>12}")

card_name                  win_count   loss_count
--------------------------------------------------
Wizard                       5138212      5141957
Valkyrie                     5039353      4931289
Zap                          4798476      4569975
Skeleton Army                4610435      4568422
The Log                      4458685      4387700
Fireball                     4156050      4215116
Hog Rider                    3661370      3547371
Mega Knight                  3556820      3536744
Arrows                       3515542      3476221
Baby Dragon                  3069898      3026894
CPU times: total: 250 ms
Wall time: 21min 16s


# 4. add win ratio

In [7]:
%%time
def add_win_ratio(row):
    card_name, win_count, loss_count = row
    total = win_count + loss_count
    win_ratio = round(win_count / total, 3) if total > 0 else 0.0
    return (card_name, win_count, loss_count, win_ratio)

card_stats_rdd = card_stats_rdd.map(add_win_ratio)
card_stats_rdd.cache()

# Show top 10 (mirroring df_card_stats.show(10))
print(f"{'card_name':<25} {'win_count':>10} {'loss_count':>12} {'win_ratio':>10}")
print("-" * 62)
for card_name, win_count, loss_count, win_ratio in card_stats_rdd.take(10):
    print(f"{card_name:<25} {win_count:>10} {loss_count:>12} {win_ratio:>10}")

card_name                  win_count   loss_count  win_ratio
--------------------------------------------------------------
Wizard                       5138212      5141957        0.5
Valkyrie                     5039353      4931289      0.505
Zap                          4798476      4569975      0.512
Skeleton Army                4610435      4568422      0.502
The Log                      4458685      4387700      0.504
Fireball                     4156050      4215116      0.496
Hog Rider                    3661370      3547371      0.508
Mega Knight                  3556820      3536744      0.501
Arrows                       3515542      3476221      0.503
Baby Dragon                  3069898      3026894      0.504
CPU times: total: 46.9 ms
Wall time: 1min 26s


# 5. results

In [8]:
%%time
# Sort by win_ratio descending and print all cards
all_stats = card_stats_rdd.sortBy(lambda x: -x[3]).collect()

print(f"{'card_name':<25} {'win_count':>10} {'loss_count':>12} {'win_ratio':>10}")
print("-" * 62)
for card_name, win_count, loss_count, win_ratio in all_stats:
    print(f"{card_name:<25} {win_count:>10} {loss_count:>12} {win_ratio:>10}")

card_name                  win_count   loss_count  win_ratio
--------------------------------------------------------------
Mega Minion                   966854       898319      0.518
Battle Ram                    509974       476804      0.517
Lava Hound                    311339       292324      0.516
Night Witch                   938355       888178      0.514
Golem                        1160228      1102175      0.513
Zap                          4798476      4569975      0.512
Lightning                     992210       946387      0.512
Goblin Gang                  2318752      2216543      0.511
P.E.K.K.A                    1941730      1861776      0.511
Flying Machine                341577       327391      0.511
Bowler                        309926       296324      0.511
Rascals                       185359       177601      0.511
Mini P.E.K.K.A               2705738      2597622       0.51
Dark Prince                  1193588      1148216       0.51
Goblin Barrel         